# Tools in LLMs are external functions you give the model so it can do things beyond just generating text. When you define a tool ─
  (like get_weather(city)), you're telling the model "you can call this function." During a conversation, if the model decides it     
  needs information or an action, it doesn't guess — it outputs a structured tool call with the function name and arguments. LangChain
   catches that, runs the actual function, and feeds the result back to the model. The model then uses that real data to give you a   
  final answer. This is how agents work — the model thinks, calls tools, observes results, and responds with actual facts instead of  
  hallucinating.

In [25]:
import os 
from dotenv import load_dotenv  
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.3-70b-versatile")
response = model.invoke("why do parrots talk?")
response.content

"Parrots are known for their remarkable ability to mimic human speech and other sounds they hear in their environment. But why do they do it? The answer lies in their evolution, social behavior, and communication needs.\n\n**Evolutionary Advantage:**\nIn the wild, parrots use vocalizations to communicate with each other, warning calls to alert other parrots to predators, and contact calls to maintain social bonds. Their ability to mimic other sounds, including those of other birds, animals, and even man-made noises, may have evolved as a way to:\n\n1. **Imitate predators**: By mimicking the calls of predators, parrots may be able to scare away potential threats or warn other parrots of danger.\n2. **Attract mates**: Male parrots may use their vocal mimicry skills to attract females, showcasing their intelligence, creativity, and social status.\n3. **Establish dominance**: Parrots may use vocalizations to establish dominance within their flock or warn other parrots to stay away from the

In [26]:
# without api call

from langchain.tools import tool

@tool
def get_wheather(location:str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


models_with_tools = model.bind_tools([get_wheather])

In [27]:
# with api call 

import requests                                                                                                                   
from langchain.tools import tool
                                                                                                                                    
@tool           
def get_weather(location: str) -> str:
      """Get the current weather at a location"""
      response = requests.get(f"https://wttr.in/{location}?format=3")
      return response.text
                                                                                                                                    
model_with_tools = model.bind_tools([get_weather])

In [28]:
response = model_with_tools.invoke("what is the weather in gujarat india")                                                        
                                                                                                                                    
# Execute each tool call and collect results                                                                                      
messages = [{"role": "user", "content": "what is the weather in gujarat india"}, response]                                        
                                                                                                                                    
for tool_call in response.tool_calls:                                                                                             
    result = get_weather.invoke(tool_call["args"])  # actually call the function                                                  
    messages.append({"role": "tool", "tool_call_id": tool_call["id"], "content": result})                                         
                  
  # Get final answer with the tool result                                                                                           
final_response = model_with_tools.invoke(messages)
print(final_response.content)

The current weather in Gujarat, India is sunny with a temperature of 29°C.


In [29]:
# Step 1: Model analyzes the user's question and decides which tool to call                                                       
  # The model doesn't run the tool — it just outputs a structured request                                                           
  # saying "I want to call get_weather with these args"                                                                             
  response = model_with_tools.invoke("what is the weather in gujarat india")                                                        
  print(response)                                                                                                                   
  # At this point, content is empty because the model chose to call a tool
  # instead of generating text. It's saying: "I need more info first"                                                               
                                                                                                                                    
  # Step 2: You inspect what the model wants to do                                                                                  
  # This lets you log, validate, or approve before actually running anything                                                        
  for tool_call in response.tool_calls:                                                                                             
      # tool_call["name"] = which function the model wants to call                                                                  
      # tool_call["args"] = what arguments it wants to pass                                                                         
      print(f"tool : {tool_call['name']}")   # e.g. "get_weather"                                                                   
      print(f"Args : {tool_call['args']}")   # e.g. {"location": "Gujarat, India"}                                                  
                                                                                                                                    
  # Step 3: You decide whether to run it
  for tool_call in response.tool_calls:                                                                                             
      print(f"Model wants to call: {tool_call['name']}({tool_call['args']})")
      approve = input("Approve? (y/n): ")                                                                                           
      if approve == "y":
          result = get_weather.invoke(tool_call["args"])                                                                            
          print(result)

  create_agent = automatic, fast, good for most cases
  bind_tools + manual loop = controlled, good for production with guardrails

IndentationError: unexpected indent (1629313702.py, line 4)

### create_agent handles everything automatically — tool calling, execution, and final response. No need for bind_tools at all

In [ ]:
### with short version of agent

import requests                                                                                                                   
from langchain.tools import tool                                                                                                  
from langchain.agents import create_agent                                                                                         
                                                                                                                                    
@tool                                                                                                                             
def get_weather(location: str) -> str:                                                                                            
      """Get the current weather at a location"""                                                                                   
      response = requests.get(f"https://wttr.in/{location}?format=3")                                                               
      return response.text                                                                                                          
                                                                                                                                    
agent = create_agent(                                                                                                             
      model="groq:llama-3.3-70b-versatile",                                                                                         
      tools=[get_weather]                                                                                                           
)                                                                                                                                 
                                                                                                                                    
response = agent.invoke({"messages": [{"role": "user", "content": "what is the weather in new york"}]})                      
print(response["messages"][-1].content)

The current weather in New York is 23°C with a sunny sky.


### tool execution loops

In [ ]:
# step - 1 model generates tool calls 
from langchain_core.messages import HumanMessage                                                                                  
                                                                                                                                    
messages = [HumanMessage(content="what is the weather in germany?")]                                                              
ai_msg = model_with_tools.invoke(messages)                                                                                        
messages.append(ai_msg)

# step-2 execute tools calls and collect results
for tool_call in ai_msg.tool_calls:
    print(f"Model wants to call: {tool_call['name']}({tool_call['args']})")
    approve = input("Approve? (y/n): ")                                                                                           
    if approve == "y":
        result = get_weather.invoke(tool_call["args"])                                                                            
        print(result)

#step -3 pass the tool results back to the model to get final answer
final_response = model_with_tools.invoke(messages)
print(final_response.content)

Model wants to call: get_weather({'location': 'Germany'})
germany: 🌦️  +23°C




In [ ]:
messages

[HumanMessage(content='what is the weather in germany?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6hfwadgqs', 'function': {'arguments': '{"location":"Germany"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 221, 'total_tokens': 235, 'completion_time': 0.054732064, 'completion_tokens_details': None, 'prompt_time': 0.022273988, 'prompt_tokens_details': None, 'queue_time': 0.160948142, 'total_time': 0.077006052}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7931-83cf-71c0-a537-28148de0fc6b-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Germany'}, 'id': '6hfwadgqs', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 14, 'total_t